In [10]:
from hdf5storage import loadmat, savemat
import numpy as np
import pickle
import os

In [11]:
def best_ratio_item(items, obj_to_max):
    ratios = items[:, obj_to_max] / items[:, -1]
    sort_indices = np.argsort(ratios)
    best_item_index = sort_indices[-1]
    return best_item_index

In [12]:
def create_heuristic_samples(pop_size, items,n_items, n_selected, n_obj, n_con, capacity, rng):
    pop_count = 0
    population = np.zeros((pop_size, n_selected), dtype=np.int32)
    objectives = np.zeros((pop_size, n_obj))
    # seen_obj = set()  
    while pop_count < pop_size:
        remain_ind = np.arange(n_items)
        knapsack_indices = np.zeros(n_selected, dtype=int)    
        knapsack = np.zeros((n_selected, (n_obj+n_con)))
        for n in range(0, n_selected):
            current_items = items[remain_ind, :]
            obj_to_max = rng.randint(n_obj) 
            choice_ind = best_ratio_item(current_items, obj_to_max)
            choice_ind_ori = remain_ind[choice_ind]
            knapsack_indices[n] = choice_ind_ori
            knapsack[n, :] = items[choice_ind_ori, :]
            remain_ind = np.delete(remain_ind, choice_ind) 
        
        constraint = np.sum(items[knapsack_indices, -1])
        if constraint <= capacity:
            # obj_vec = np.sum(knapsack[:, :n_obj], axis=0)
            # obj_key = tuple(obj_vec)
            # if obj_key not in seen_obj:
                # seen_obj.add(obj_key)
            population[pop_count, :] = knapsack_indices
            objectives[pop_count, :] = np.sum(knapsack[:, :n_obj], axis=0)
            pop_count += 1
    
    return population, objectives

In [13]:
from pymoo.indicators.hv import HV

def calculate_hv(obj1, obj2, n_obj):
    A = obj1.astype(np.float64)
    B = obj2.astype(np.float64)
    A_min = -A
    B_min = -B

    ref = np.zeros(n_obj)
    hv = HV(ref_point=ref)

    A_hv = hv(A_min)
    B_hv = hv(B_min)

    # print(A_hv)
    # print(B_hv)
    # print((A_hv-B_hv)/B_hv)

    return A_hv, B_hv

In [14]:
def calculate_dominated(obj1, obj2, n_obj):
    pf_predicted = obj1
    pf_actual = obj2
    dominated = np.zeros(len(pf_predicted))
    for i in range(len(pf_predicted)):
        for j in range(len(pf_actual)):
            if np.all(pf_actual[j, :n_obj] >= pf_predicted[i, :n_obj]) and \
                np.any(pf_actual[j, :n_obj] > pf_predicted[i, :n_obj]):
                dominated[i] = 1
                break
    return np.sum(dominated), np.sum(dominated)/len(pf_predicted)

In [15]:
# items = kn['items'][0]
# population, objectives = create_heuristic_samples(pop_size, items,n_items, n_selected, n_obj, n_con, capacity, rng)
# objectives_unique = np.unique(objectives, axis=0)

# with open(f'final_2_3_120_12_obj3_run{0}.pkl', 'rb') as f:
#     results = pickle.load(f)
# converged_pf = results['converged_pf_table'][-1]

# num_obju = objectives_unique.shape[0]
# num_cpf = converged_pf.shape[0]

# hv_obju, hv_cpf = calculate_hv(objectives_unique, converged_pf, n_obj)

# num_dominated_obju, ratio_dominated_obju = calculate_dominated(objectives_unique, converged_pf, n_obj)
# num_dominated_cpf, ratio_dominated_cpf = calculate_dominated(converged_pf, objectives_unique, n_obj)

In [16]:
# print(num_obju)
# print(num_cpf)
# print(hv_obju)
# print(hv_cpf)
# print(num_dominated_obju)
# print(ratio_dominated_obju)
# print(num_dominated_cpf)
# print(ratio_dominated_cpf)

In [17]:
n_items = 120
n_selected = 12
n_obj = 2

In [18]:
kn = loadmat(f'/data/knapsack/runA/kn_2_3_allneg_{n_items}_{n_selected}_{n_obj}.mat')
n_con = kn['n_con'] 
capacity = kn['capacity']

In [19]:
rng = np.random.RandomState(1123)
pop_size = 500_000

In [20]:
for i in range(20):
    items = kn['items'][i]
    population, objectives = create_heuristic_samples(pop_size, items,n_items, n_selected, n_obj, n_con, capacity, rng)
    objectives_unique = np.unique(objectives, axis=0)
    
    with open(f'final_2_3_{n_items}_{n_selected}_obj{n_obj}_run{i}.pkl', 'rb') as f:
        results = pickle.load(f)
    converged_pf = results['converged_pf_table'][-1]

    num_obju = objectives_unique.shape[0]
    num_cpf = converged_pf.shape[0]

    hv_obju, hv_cpf = calculate_hv(objectives_unique, converged_pf, n_obj)

    num_dominated_obju, ratio_dominated_obju = calculate_dominated(objectives_unique, converged_pf, n_obj)
    num_dominated_cpf, ratio_dominated_cpf = calculate_dominated(converged_pf, objectives_unique, n_obj)
    
    final_results = {
        'num_obju': num_obju,
        'num_cpf': num_cpf,
        'hv_obju': hv_obju,
        'hv_cpf': hv_cpf,
        'num_dominated_obju': num_dominated_obju,
        'ratio_dominated_obju': ratio_dominated_obju,
        'num_dominated_cpf': num_dominated_cpf,
        'ratio_dominated_cpf': ratio_dominated_cpf,
    }

    file = f'heuristic_2_3_{n_items}_{n_selected}_obj{n_obj}_run{i}.pkl'
    if os.path.exists(file):
        print(f"File {file} already exists")
    else:   
        with open(file, 'wb') as f:
            pickle.dump(final_results, f)

In [21]:
# objectives_unique = np.unique(objectives, axis=0)# 

In [22]:
# with open(f'final_2_3_120_12_obj4_run{run}.pkl', 'rb') as f:
#     results = pickle.load(f)
# converged_pf = results['converged_pf_table'][-1]

In [23]:
# from matplotlib import pyplot as plt
# %matplotlib widget
# from mpl_toolkits.mplot3d import Axes3D

# fig = plt.figure(figsize=(10, 10))
# ax = fig.add_subplot(projection='3d')
# ax.plot(objectives_unique[:,0], objectives_unique[:,1], objectives_unique[:,2], marker='o', linestyle='None',
#         markerfacecolor='none', markeredgecolor='k',
#         markersize=5)
# ax.plot(converged_pf[:,0], converged_pf[:,1], converged_pf[:,2], 'rs', alpha=0.3, markersize=5)
# # ax.plot(pareto_front[:,0], pareto_front[:,1], pareto_front[:,2], 'go', alpha=0.3, markersize=5)
# plt.show()

In [24]:
# obj1 = objectives_unique[:, :n_obj]
# obj2 = converged_pf[:, :n_obj]

In [25]:
# def eval_knapsack(knapsack,pf_actual,n_obj,scale,shape):
#     pf_predicted = knapsack
#     std_dev = np.sqrt(shape)*scale
#     count = 0 
#     pfdif = np.zeros(pf_actual.shape[0])
#     for j in range(pf_actual.shape[0]):
#         if np.any(pf_predicted[0:n_obj] > pf_actual[j,0:n_obj]):
#             count = count + 1
#         pfdif[j] = np.linalg.norm(pf_predicted[0:n_obj]/std_dev[0:n_obj] - pf_actual[j,0:n_obj]/std_dev[0:n_obj])
#     pf_difference = np.min(pfdif) 
#     domination_score = 1-count/pf_actual.shape[0]
#     return domination_score, pf_difference

In [26]:
# pf_difference = np.zeros(len(obj2))
# domination_score = np.zeros(len(obj2))
# for kk in range(len(obj2)):
#     domination_score[kk], pf_difference[kk] = eval_knapsack(obj2[kk], obj1, n_obj, kn['scale'],kn['shape'])